# LoRA Training — Wan 2.1 I2V 480P
**วิธีใช้:**
1. Runtime → Change runtime type → T4 GPU
2. วาง clips ไว้ใน Google Drive ที่ `MyDrive/COMP_ANI/clips/`
3. รันทีละ cell ตามลำดับ

> ⚠️ Free Colab disconnect ทุก ~12 ชม. — ถ้า training ยังไม่เสร็จต้อง resume จาก checkpoint

In [ ]:
# CELL 1 — เช็ค GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# เช็คว่าเจอ clips
import os
CLIPS_PATH = '/content/drive/MyDrive/COMP_ANI/clips'
clips = [f for f in os.listdir(CLIPS_PATH) if f.endswith('.mp4')]
print(f'Found {len(clips)} clips')

In [ ]:
# CELL 3 — Clone repo
!git clone https://github.com/nepitunepq/Animators.git /content/Animators
%cd /content/Animators

In [ ]:
# CELL 4 — Install dependencies
!pip install -q diffusers transformers accelerate huggingface_hub
!pip install -q bitsandbytes tqdm opencv-python

# Clone diffusion-pipe สำหรับ LoRA training
!git clone --recurse-submodules https://github.com/tdrussell/diffusion-pipe /content/diffusion-pipe
!pip install -q -r /content/diffusion-pipe/requirements.txt

In [ ]:
# CELL 5 — Copy clips จาก Drive มา local (เร็วกว่าอ่านจาก Drive ตรงๆ)
import shutil, os

# สร้างทั้งสอง folder เพื่อรองรับ config.py ทุก version
os.makedirs('/content/Animators/clips', exist_ok=True)
os.makedirs('/content/Animators/blender_clips', exist_ok=True)

for f in clips:
    src = os.path.join(CLIPS_PATH, f)
    shutil.copy2(src, f'/content/Animators/clips/{f}')
    shutil.copy2(src, f'/content/Animators/blender_clips/{f}')

print(f'Copied {len(clips)} clips to clips/ and blender_clips/')

In [ ]:
# CELL 6 — Prepare dataset (resize + captions)
%cd /content/Animators
!python dataset_prep.py

In [ ]:
# CELL 7 — Download Wan 2.1 T2V 1.3B (เล็กพอสำหรับ T4 free Colab)
# โมเดลนี้ขนาด ~5GB เท่านั้น เหมาะกับ T4 15GB VRAM
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='Wan-AI/Wan2.1-T2V-1.3B',
    local_dir='/content/Animators/models/wan2.1',
    ignore_patterns=['*.bin'],
)
print('Model downloaded ✓')

In [ ]:
# CELL 8 — แก้ config ให้ตรงกับ T4 + Wan 2.1
CONFIG_PATCH = '''
import sys, os
sys.path.insert(0, '/content/Animators')

# Override paths and settings for Colab T4
MODEL_DIR     = '/content/Animators/models/wan2.1'
DATASET_DIR   = '/content/Animators/wan-dataset'
LORA_OUTPUT   = '/content/drive/MyDrive/COMP_ANI/lora_output'  # save to Drive
BLENDER_CLIPS = '/content/Animators/clips'

# T4 memory limits
BATCH_SIZE  = 1
GRAD_ACCUM  = 8
NUM_FRAMES  = 17    # ลดจาก 49 → 17 เพื่อประหยัด VRAM
TARGET_W    = 480
TARGET_H    = 288
MAX_STEPS   = 1500
SAVE_EVERY  = 250
LORA_RANK   = 16
LORA_ALPHA  = 16
LEARNING_RATE = 1e-4
SEED        = 42
'''

with open('/content/Animators/config_colab.py', 'w') as f:
    f.write(CONFIG_PATCH)
print('Colab config written ✓')

In [ ]:
# CELL 9 — Run LoRA training via diffusion-pipe TOML config
import os, subprocess, sys
sys.path.insert(0, '/content/Animators')
from config_colab import *

VIDEO_DIR   = os.path.join(DATASET_DIR, 'videos')
CAPTION_DIR = os.path.join(DATASET_DIR, 'captions')
os.makedirs(LORA_OUTPUT, exist_ok=True)

# สร้าง TOML config ที่ diffusion-pipe ต้องการ
TOML = f"""
output_dir = '{LORA_OUTPUT}'

[model]
model_path = '{MODEL_DIR}'
model_type = 'wan'
dtype = 'bfloat16'

[network]
type = 'lora'
rank = {LORA_RANK}
alpha = {LORA_ALPHA}

[optimizer]
type = 'adamw8bit'
lr = {LEARNING_RATE}

[training]
steps = {MAX_STEPS}
batch_size = {BATCH_SIZE}
gradient_accumulation_steps = {GRAD_ACCUM}
gradient_checkpointing = true
save_every_n_steps = {SAVE_EVERY}
seed = {SEED}

[[datasets]]
type = 'video'
directory = '{VIDEO_DIR}'
caption_extension = '.txt'
resolution = [{TARGET_W}, {TARGET_H}]
num_frames = {NUM_FRAMES}
fps = 24
"""

config_path = '/content/Animators/lora_config.toml'
with open(config_path, 'w') as f:
    f.write(TOML)
print('TOML config written ✓')
print(TOML)

# รัน training
print('Starting training...')
result = subprocess.run(
    ['python', '/content/diffusion-pipe/train.py', '--config', config_path],
    check=True
)

In [ ]:
# CELL 10 — ดู loss curve (รันระหว่าง training หรือหลัง training)
import glob, json
import matplotlib.pyplot as plt

log_files = sorted(glob.glob(f'{LORA_OUTPUT}/logs/*.jsonl'))
if not log_files:
    print('ยังไม่มี log — training ยังไม่เริ่มหรือยังไม่ถึง step แรก')
else:
    steps, losses = [], []
    with open(log_files[-1]) as f:
        for line in f:
            d = json.loads(line)
            if 'train_loss' in d:
                steps.append(d['step'])
                losses.append(d['train_loss'])

    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, color='purple')
    plt.axhline(0.02, color='green', linestyle='--', label='Target < 0.02')
    plt.xlabel('Steps'); plt.ylabel('Loss')
    plt.title('LoRA Training Loss')
    plt.legend(); plt.grid(alpha=0.3)
    plt.show()
    print(f'Latest loss: {losses[-1]:.4f} at step {steps[-1]}')